<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/seq2one/stage_07_05_lstm_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_05 -  Modelo LSTM (many-to-one)**

El **LSTM (Long Short-Term Memory)** es una red neuronal recurrente diseñada para modelar **dependencias temporales** en secuencias.

A diferencia de los modelos seq2one con ventanas aplanadas, el LSTM procesa la
secuencia **minuto a minuto**, manteniendo un estado interno que resume la
dinámica temporal pasada.

En este pipeline se utiliza en configuración **many-to-one**:
- **Entrada:** secuencia histórica (60 × 20).
- **Salida:** un único valor escalar futuro.

El LSTM introduce **memoria temporal explícita**, siendo el primer modelo capaz
de explotar directamente la estructura secuencial intradía del problema.


# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [2]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows_seq2one/train_delta60_ws60.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows_seq2one/test_delta60_ws60.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows_seq2one/valid_delta60_ws60.npz"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows_seq2one/train_delta90_ws60.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows_seq2one/test_delta90_ws60.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows_seq2one/valid_delta90_ws60.npz"))

#ESCALADOR GLOBAL
IN_SCALER = Path(os.environ.get("IN_SCALER", "data/scaled/scaler.joblib"))



In [3]:
#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
#IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
#IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [5]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z

IN_SCALER = DRIVE_DIR / IN_SCALER

#IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
#IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

## **2. Reproducibilidad**

In [6]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [7]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [8]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [9]:
# Construye un StageConfig leyendo ambos reports.
SUMMARY = """
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )
"""

In [10]:
#states_h60 = load_state_from_reports(horizon=60)
#states_h60

In [11]:
#states_h90 = load_state_from_reports(horizon=90)
#states_h90


## **4. Importar métricas comunes desde .py**

In [12]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [13]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **5. Carga de data windows**

In [14]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [15]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [16]:
# Cargar scaler
#scaler = joblib.load("/content/drive/MyDrive/neural_profit/data/scaled/scaler.joblib")

In [17]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [18]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


In [19]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (330144, 1200) (330144,)
H60 Valid: (70952, 1200) (70952,)
H60 Test : (70590, 1200) (70590,)
H90 Train: (330144, 1200) (330144,)
H90 Valid: (70952, 1200) (70952,)
H90 Test : (70590, 1200) (70590,)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **6. Sanity Check**

In [20]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [21]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [22]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [23]:
summary = run_sanity_checks_all_horizons_seq2one(bundle_60, bundle_90)

#summary["h60"]["train"]

[sanity_check_seq2one] train_h60 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=57.959399
[sanity_check_seq2one] valid_h60 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=92.974502
[sanity_check_seq2one] test_h60 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=54.347220
OK h60 (h=60)
[sanity_check_seq2one] train_h90 | X=(330144, 1200) | y=(330144,) | mode=2d | y_std=70.925909
[sanity_check_seq2one] valid_h90 | X=(70952, 1200) | y=(70952,) | mode=2d | y_std=114.837946
[sanity_check_seq2one] test_h90 | X=(70590, 1200) | y=(70590,) | mode=2d | y_std=67.180748
OK h90 (h=90)


# **DEFINICIÓN DE MODELO**

## **7. Definición del modelo — placeholder**

### **7.1. Modelo LSTM - many to one**

**Idea básica**

El **LSTM (Long Short-Term Memory)** es una red neuronal recurrente diseñada para
modelar **dependencias temporales** en secuencias, manteniendo un estado interno
que permite recordar información relevante a lo largo del tiempo.

A diferencia del MLP, el LSTM **no aplana la ventana**, sino que procesa la
secuencia histórica **paso a paso**, preservando el orden temporal de los datos.

En configuración **many-to-one**, el modelo recibe una secuencia histórica
y produce un único valor escalar futuro.

Formalmente, el modelo puede expresarse como:

$$
h_t = \mathrm{LSTM}(x_t, h_{t-1})
$$

$$
\hat{y}_t = W_o h_T + b_o
$$

donde:
- $x_t \in \mathbb{R}^{20}$ es el vector de features en el minuto $t$,
- $h_t$ es el estado oculto del LSTM,
- $h_T$ resume toda la ventana histórica (por ejemplo, 60 minutos),
- $W_o, b_o$ son los parámetros de la capa de salida.

---

**Regularización (LSTM)**

**Riesgo:** Medio–alto, debido a la capacidad del modelo y a su memoria temporal.

La regularización **no es automática** y debe controlarse explícitamente:

- **Early stopping:**
  - Mecanismo principal para evitar sobreajuste.
- **Control del tamaño del estado oculto:**
  - Hidden size moderado.
- **Número de capas limitado:**
  - 1 (máximo 2) capas LSTM.
- **Dropout (opcional):**
  - Aplicado entre capas, no dentro de la recurrencia.

La regularización en LSTM es principalmente **estructural y temporal**, más que
puramente paramétrica.

---

**Por qué el LSTM es relevante en este proyecto**

- Entrada **secuencial explícita**: 60 × 20 (minutos × features).
- Capacidad para capturar:
  - dependencias temporales,
  - dinámica intradía,
  - patrones que no son accesibles a modelos aplanados.
- Modelo:
  - más expresivo que MLP,
  - más alineado con la naturaleza temporal del problema.

El LSTM es el **primer modelo del pipeline que explota directamente la estructura
temporal**, marcando la transición desde enfoques estáticos (seq2one aplanado)
hacia modelos verdaderamente secuenciales.

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Tipo: LSTM many-to-one
- Número de capas: 1
- Dimensión del estado oculto: moderada (por ejemplo, 64–128)
- Dropout: desactivado inicialmente
- Optimización: Adam
- Early stopping: activado
- **Sin validación interna automática** (la evaluación se realiza externamente en VALID)

El ajuste fino de la arquitectura y la regularización se aborda en etapas posteriores.


### **7.2. Imports (PyTorch) + semillas**

In [25]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### **7.3. Utilidad: reshape de X desde (n, 1200) a (n, 60, 20)**

In [24]:
def reshape_X_flat_to_seq(X_flat: np.ndarray, *, seq_len: int = 60, n_features: int = 20) -> np.ndarray:
    """
    Convierte X de (n, flat_dim) a (n, seq_len, n_features).
    Espera flat_dim = seq_len * n_features.
    """
    X_flat = np.asarray(X_flat, dtype=np.float32)
    if X_flat.ndim != 2:
        raise ValueError(f"Se espera X 2D (n, flat_dim). Recibido: {X_flat.shape}")

    n, flat_dim = X_flat.shape
    expected = seq_len * n_features
    if flat_dim != expected:
        raise ValueError(f"flat_dim={flat_dim} != seq_len*n_features={expected} ({seq_len}*{n_features})")

    return X_flat.reshape(n, seq_len, n_features)


### **7.4. DataLoaders desde bundle (con reshape interno)**

In [26]:
def make_lstm_loaders_from_bundle(
    bundle: dict,
    *,
    seq_len: int = 60,
    n_features: int = 20,
    batch_size: int = 16384,
    num_workers: int = 0,
) -> dict:
    """
    Crea loaders train/valid/test para LSTM many-to-one.
    - X: (n, 1200) -> (n, 60, 20)
    - y: (n,) -> (n, 1)
    """
    loaders = {}
    for split in ["train", "valid", "test"]:
        X_flat = bundle[split]["X"]
        y = bundle[split]["y"]

        X = reshape_X_flat_to_seq(X_flat, seq_len=seq_len, n_features=n_features)           # (n,60,20)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)                                   # (n,1)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )
    return loaders


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaders_lstm_60 = make_lstm_loaders_from_bundle(bundle_60, seq_len=60, n_features=20, batch_size=16384)
loaders_lstm_90 = make_lstm_loaders_from_bundle(bundle_90, seq_len=60, n_features=20, batch_size=16384)


### **7.5. Modelo LSTM many-to-one**

In [28]:
class LSTMSeq2One(nn.Module):
    def __init__(self, *, n_features: int = 20, hidden_size: int = 128, num_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, (h_n, c_n) = self.lstm(x)
        last_h = h_n[-1]            # (batch, hidden_size) último layer
        y_hat = self.head(last_h)   # (batch, 1)
        return y_hat



### **7.6. Entrenamiento con early stopping (VALID)**



#### **7.6.1. Función de entrenamiento LSTM**


In [32]:
@torch.no_grad()
def eval_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)


def train_lstm(
    loaders: dict,
    *,
    n_features: int = 20,
    hidden_size: int = 128,
    num_layers: int = 1,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    device: torch.device,
) -> nn.Module:
    model = LSTMSeq2One(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in loaders["train"]:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

        valid_mse = eval_mse(model, loaders["valid"], device)
        print(f"epoch={epoch:02d} | valid_mse={valid_mse:.6f}")

        if valid_mse < best_valid - 1e-9:
            best_valid = valid_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping (patience={patience}). Best valid_mse={best_valid:.6f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


#### **7.6.2. Sanity check LSTM**


In [36]:
import torch
import torch.nn as nn

def sanity_gpu_check_lstm(loaders: dict, model: nn.Module, device: torch.device) -> None:
    # 1) Dispositivo esperado
    print("device esperado:", device)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

    # 2) Modelo en GPU
    model = model.to(device)
    print("modelo.device:", next(model.parameters()).device)

    # 3) Un batch en GPU
    xb, yb = next(iter(loaders["train"]))
    xb = xb.to(device, non_blocking=True)
    yb = yb.to(device, non_blocking=True)

    print("xb.device:", xb.device, "| xb.shape:", tuple(xb.shape))
    print("yb.device:", yb.device, "| yb.shape:", tuple(yb.shape))

    # 4) Forward en GPU
    model.train()
    out = model(xb)
    print("out.device:", out.device, "| out.shape:", tuple(out.shape))

    # 5) Loss + backward (gradientes en GPU)
    loss_fn = nn.MSELoss()
    loss = loss_fn(out, yb)
    loss.backward()

    # Verifica que haya gradientes y que estén en GPU
    grad_ok = []
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        grad_ok.append((name, p.grad.device))
    print("gradientes (ejemplos):", grad_ok[:3] if grad_ok else "NO HAY GRADIENTES")

    # 6) Memoria GPU (opcional)
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**2)
        reserved = torch.cuda.memory_reserved() / (1024**2)
        print(f"GPU mem | allocated={alloc:.1f} MB | reserved={reserved:.1f} MB")

    print("[OK] Check GPU completado.")


In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loaders_lstm_60 = make_lstm_loaders_from_bundle(bundle_60, seq_len=60, n_features=20, batch_size=16384)

test_lstm = LSTMSeq2One(n_features=20, hidden_size=128, num_layers=1, dropout=0.0)

sanity_gpu_check_lstm(loaders_lstm_60, test_lstm, device)

device esperado: cuda
CUDA disponible: True
GPU: NVIDIA L4
modelo.device: cuda:0
xb.device: cuda:0 | xb.shape: (16384, 60, 20)
yb.device: cuda:0 | yb.shape: (16384, 1)
out.device: cuda:0 | out.shape: (16384, 1)
gradientes (ejemplos): [('lstm.weight_ih_l0', device(type='cuda', index=0)), ('lstm.weight_hh_l0', device(type='cuda', index=0)), ('lstm.bias_ih_l0', device(type='cuda', index=0))]
GPU mem | allocated=93.0 MB | reserved=15706.0 MB
[OK] Check GPU completado.


#### **7.6.3. Entrenamiento LSTM**


In [38]:
lstm_60 = train_lstm(
    loaders_lstm_60,
    n_features=20,
    hidden_size=128,
    num_layers=1,
    dropout=0.0,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    device=device
    )

epoch=01 | valid_mse=8638.828504
epoch=02 | valid_mse=8639.782050
epoch=03 | valid_mse=8640.074698
epoch=04 | valid_mse=8635.803191
epoch=05 | valid_mse=8636.349053
epoch=06 | valid_mse=8630.992615
epoch=07 | valid_mse=8640.062239
epoch=08 | valid_mse=8629.891166
epoch=09 | valid_mse=8638.545918
epoch=10 | valid_mse=8650.331548
epoch=11 | valid_mse=8646.932659
epoch=12 | valid_mse=8638.774749
epoch=13 | valid_mse=8651.926711
Early stopping (patience=5). Best valid_mse=8629.891166


In [39]:
lstm_90 = train_lstm(
    loaders_lstm_90,
    n_features=20,
    hidden_size=128,
    num_layers=1,
    dropout=0.0,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    device=device
    )

epoch=01 | valid_mse=13176.245800
epoch=02 | valid_mse=13178.168452
epoch=03 | valid_mse=13178.560943
epoch=04 | valid_mse=13178.311027
epoch=05 | valid_mse=13174.533346
epoch=06 | valid_mse=13178.113203
epoch=07 | valid_mse=13168.321316
epoch=08 | valid_mse=13182.420622
epoch=09 | valid_mse=13180.677416
epoch=10 | valid_mse=13186.838426
epoch=11 | valid_mse=13161.270718
epoch=12 | valid_mse=13185.235934
epoch=13 | valid_mse=13187.075149
epoch=14 | valid_mse=13180.455237
epoch=15 | valid_mse=13146.622759
epoch=16 | valid_mse=13166.292451
epoch=17 | valid_mse=13198.625662
epoch=18 | valid_mse=13147.329744
epoch=19 | valid_mse=13193.676824
epoch=20 | valid_mse=13148.601308
Early stopping (patience=5). Best valid_mse=13146.622759


### **7.7. Predicciones LSTM**


In [42]:
@torch.no_grad()
def predict_lstm(model: nn.Module, X_flat: np.ndarray, *, device: torch.device, seq_len: int = 60, n_features: int = 20, batch_size: int = 16384) -> np.ndarray:
    model.eval()
    X_seq = reshape_X_flat_to_seq(X_flat, seq_len=seq_len, n_features=n_features)  # (n,60,20)
    n = X_seq.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X_seq[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb).squeeze(-1)  # (batch,)
        preds.append(yb.detach().cpu().numpy())

    return np.concatenate(preds, axis=0)



In [43]:
# H=60
y_pred_valid_60 = predict_lstm(lstm_60, bundle_60["valid"]["X"], device=device, seq_len=60, n_features=20)
y_pred_test_60  = predict_lstm(lstm_60, bundle_60["test"]["X"],  device=device, seq_len=60, n_features=20)

# H=90
y_pred_valid_90 = predict_lstm(lstm_90, bundle_90["valid"]["X"], device=device, seq_len=60, n_features=20)
y_pred_test_90  = predict_lstm(lstm_90, bundle_90["test"]["X"],  device=device, seq_len=60, n_features=20)


## **8. Métricas ML**

In [44]:
import pandas as pd

def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])


In [45]:
# ============================================================
# H=60
# ============================================================
y_valid_60 = bundle_60["valid"]["y"]
y_test_60  = bundle_60["test"]["y"]

metrics_valid_60 = compute_seq2one_metrics(y_valid_60, y_pred_valid_60, compute_r2=True)
metrics_test_60  = compute_seq2one_metrics(y_test_60,  y_pred_test_60,  compute_r2=True)

df_valid_60 = metrics_to_df(metrics_valid_60, model="lstm", split="valid", horizon=60)
df_test_60  = metrics_to_df(metrics_test_60,  model="lstm", split="test",  horizon=60)

# ============================================================
# H=90
# ============================================================
y_valid_90 = bundle_90["valid"]["y"]
y_test_90  = bundle_90["test"]["y"]

metrics_valid_90 = compute_seq2one_metrics(y_valid_90, y_pred_valid_90, compute_r2=True)
metrics_test_90  = compute_seq2one_metrics(y_test_90,  y_pred_test_90,  compute_r2=True)

df_valid_90 = metrics_to_df(metrics_valid_90, model="lstm", split="valid", horizon=90)
df_test_90  = metrics_to_df(metrics_test_90,  model="lstm", split="test",  horizon=90)

In [46]:
# ============================================================
# Tabla final (VALID+TEST, H60+H90)
# ============================================================
df_lstm_metrics = pd.concat([df_valid_60, df_valid_90, df_test_60, df_test_90], ignore_index=True)
df_lstm_metrics = df_lstm_metrics.sort_values(["split", "horizon_min"]).reset_index(drop=True)

df_lstm_metrics

,model,split,horizon_min,MAE,RMSE,R2,DA
0,lstm,test,60,40.269801,54.474356,-0.004684,0.472238
1,lstm,test,90,48.978666,67.256351,-0.002252,0.460856
2,lstm,valid,60,61.878273,92.897208,0.001662,0.483093
3,lstm,valid,90,75.932251,114.658722,0.003119,0.481128


## **9. Guardar artefactos para Stage_08**

In [47]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"metrics_{name}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path


In [48]:
save_seq2one_metrics(
    df_lstm_metrics,
    name="lstm",
)

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_lstm.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics/metrics_lstm.parquet')

## **10. Resultados y conclusiones parciales — MLP vs Lasso (seq2one)**

**1. Comparación directa MLP vs Lasso (VALID)**

> **El conjunto VALID es el criterio principal de comparación.**

- **Horizonte 60 (VALID)**

  | Modelo | MAE | RMSE | R² | DA |
  |------|-----|------|----|----|
  | **MLP** | 61.86 | 93.01 | -0.0008 | 0.4779 |
  | **Lasso** | **61.56** | **92.96** | **0.0002** | 0.4766 |

  - **Lasso supera levemente al MLP** en MAE, RMSE y R².  
  - La métrica direccional (DA) es prácticamente idéntica.

- **Horizonte 90 (VALID)**

  | Modelo | MAE | RMSE | R² | DA |
  |------|-----|------|----|----|
  | **MLP** | 76.15 | 114.89 | -0.0009 | 0.4761 |
  | **Lasso** | **75.89** | **114.86** | **-0.0003** | 0.4750 |

  - **Lasso vuelve a mostrar un desempeño ligeramente superior**, aunque con diferencias marginales.

**2. Lectura de los resultados**

- El **MLP no mejora al modelo Lasso** en ninguno de los horizontes evaluados.
- Ambos modelos presentan métricas muy cercanas entre sí y al baseline lineal.
- El coeficiente de determinación **R² ≈ 0** en todos los casos, lo que indica que:
  - la varianza explicada es prácticamente nula,
  - no se está capturando una señal predictiva fuerte.

Esto **no representa un error de implementación** ni de entrenamiento, sino un
resultado informativo sobre la naturaleza del problema.

**3. Conclusión técnica**

> **La incorporación de no linealidad mediante un MLP feedforward no aporta valor frente a un modelo lineal regularizado (Lasso) en el enfoque seq2one actual.**

Este comportamiento sugiere, en orden de probabilidad, que:

1. La **señal predictiva es débil** para estos horizontes con los features actuales.
2. La información relevante ya está **capturada linealmente**.
3. El problema requiere **modelos que exploten explícitamente la estructura temporal**,
   más allá de ventanas aplanadas.

**4. Decisión para el pipeline**

  - **Lasso** se mantiene como la **mejor referencia lineal**.
  - **MLP no justifica su mayor complejidad** en esta etapa.
  - Incrementar capacidad feedforward (más capas, más neuronas, más epochs)
    **no es la vía correcta** para mejorar el desempeño.

**5. Próximo paso recomendado**

El siguiente avance lógico **NO** consiste en:

- mayor tuning del MLP,
- arquitecturas feedforward más profundas.

El siguiente paso lógico **SÍ** es avanzar hacia modelos que respeten la
**estructura temporal intrínseca** del problema:

- LSTM / GRU (many-to-one),
- TCN (many-to-one).

En ese punto recién es razonable esperar una mejora sustantiva en desempeño
predictivo intradía.
